In [1]:
from pathlib import Path
import pickle
import sys
import os
sys.path.append('../')

DATA_DIR = Path('../data/100000_data_points')
DATA_PATH = DATA_DIR / "synthetic_rl_datasets.pkl"
try:
    with open(DATA_PATH, 'rb') as f:
        all_data = pickle.load(f)
except FileNotFoundError:
    print(f"File not found: {DATA_PATH}")
except Exception as e:
    print(f"Error loading dataset: {e}")


In [2]:
import random
import sys
import torch
from tqdm import tqdm
import os
import matplotlib.pyplot as plt

sys.path.append('../')
# Add the root directory to the Python path to find your modules

from models.reward_cnn import RewardCNNCentralized
from tests.cnn.centralized.generate_synthetic_states import (
    generate_synthetic_state_at_most_1_apples,
)
from config import MODEL_DIR, GRAPHS_DIR
import torch
from tests.cnn.centralized.generate_synthetic_states import (
    generate_synthetic_state_at_most_1_apples,
)
from train_scripts.train_centralized_cnn import MODEL_SAVE_PATH
import matplotlib.pyplot as plt
from config import GRAPHS_DIR, OUT_DATA_DIR
from tqdm import tqdm


keys=list(all_data.keys())
hidden_dim = [8, 32, 64, 256]
print(f"Available keys in dataset: {keys}")
p_pick_apple = 0.5


--- PyTorch is configured to use: cuda ---
Available keys in dataset: ['6x6_2_agents', '9x9_4_agents', '12x12_7_agents']


In [ ]:
import json
from models.reward_network import RewardNetwork
from helpers.controllers import ViewController

viewController = ViewController(vision=0, new_input=True)
def processed_state(state):
    return viewController.state_to_nn_input(state, None, None)

for key in keys:
    accuracies = []
    print(f"--------------Processing key: {key} --------------")
    for hidden in hidden_dim:
        print(f"--- Hidden features: {hidden} ---")
        data = all_data[key]

        NUM_DATA_POINTS = 100000


        num_picks_apple = int(NUM_DATA_POINTS * p_pick_apple)
        num_no_picks_apple = NUM_DATA_POINTS - num_picks_apple

        data_with_apple = data['picks_apple'][:num_picks_apple]
        data_without_apple = data['doesnt_pick_apple'][:num_no_picks_apple]

        state_agentpos_reward = data_with_apple + data_without_apple
        random.shuffle(state_agentpos_reward)
        print(len(state_agentpos_reward))
        print(state_agentpos_reward[0])

        NUM_TRAIN_EPISODES = 1000
        percent_train = 0.8
        total_train = int(0.8 * NUM_DATA_POINTS)
        state_agentpos_reward_TRAIN = state_agentpos_reward[:total_train]
        state_agentpos_reward_TEST = state_agentpos_reward[total_train:]
        if key == '6x6_2_agents':
            WIDTH = 6
            HEIGHT = 6
            NUM_AGENTS = 2
        elif key == '9x9_4_agents':
            WIDTH = 9
            HEIGHT = 9
            NUM_AGENTS = 4
        elif key == '12x12_7_agents':
            WIDTH = 12
            HEIGHT = 12
            NUM_AGENTS = 7
        else:
            raise ValueError(f"Unknown key: {key}")
        BATCH_SIZE = 32
        lr = 0.01

        model = RewardNetwork(WIDTH * HEIGHT * 2, 1, lr, 0.99, hidden, 4) # parameters taken from reward_learning.py
        # --- Training Loop ---
        for i in tqdm(range(NUM_TRAIN_EPISODES), desc="Training"):
            for b in range(BATCH_SIZE):
                row_index = i * BATCH_SIZE + b
                state = state_agentpos_reward_TRAIN[row_index]["state"]
                label = state_agentpos_reward_TRAIN[row_index]["reward"]
                agent_pos = state_agentpos_reward_TRAIN[row_index]["agent_pos"]
                model.add_experience(processed_state(state), label)
            model.train()
        print(f"Final loss after training: {model.loss_history[-1]}")
        
        # plot model loss
        plt.plot(model.loss_history)
        plt.xlabel("Training Iterations")
        plt.ylabel("Loss")
        plt.title("Model Loss Over Time")
        plt.savefig(GRAPHS_DIR / f"mlp_centralized_loss_{key}_{hidden}h.png")



        # --- Configuration (should match the trained model) ---
        num_test_episodes = len(state_agentpos_reward_TEST)
        # --- Test Loop (no training here!) ---
        num_correct = 0
        num_correct_when_reward_1 = 0
        num_correct_when_reward_0 = 0
        num_reward_1 = 0
        num_reward_0 = 0
        tol = 10e-2
        for i in tqdm(range(num_test_episodes), leave=False):
            state = state_agentpos_reward_TEST[i]["state"]
            label = state_agentpos_reward_TEST[i]["reward"]
            agent_pos = state_agentpos_reward_TEST[i]["agent_pos"]
            # Get the raw float prediction
            prediction = model.get_model_reward_prediction(processed_state(state)).item()
            error = abs(prediction - label)
            if error < tol:
                num_correct += 1
        accuracy = num_correct / num_test_episodes
        accuracies.append(accuracy)
    # save the accuracies for this key to a file as raw data
    filename = OUT_DATA_DIR / f"mlp_centralized_accuracies_{key}.txt"
    with open(filename, "w") as f:
        json.dump(accuracies, f, indent=4)


--------------Processing key: 6x6_2_agents --------------
--- Hidden features: 8 ---
100000
{'state': {'apples': array([[0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0]], dtype=int8), 'agents': array([[0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 1, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [1, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0]], dtype=int8)}, 'agent_pos': (np.int64(2), np.int64(2)), 'reward': 0}


Training:   0%|          | 0/1000 [00:41<?, ?it/s]


TypeError: unsupported operand type(s) for //: 'NoneType' and 'int'